# 10.8 原本的文字功能還能正常嗎？

# 第 10 章：圖片如何進入文字模型

前置：embedding、序列與Linear。圖片入口先用16×16合成RGB，不需外部權重；圖片是否被理解必須等語言／視覺基礎能力訓練後測試。程式在tiny_perceptron/multimodal.py。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：圖片切小方塊：先檢查能不能拼回來**

16×16 RGB，patch邊長4，得到16個patch。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/patchify.svg")))

**先想一想：**新包裝即使不提供圖片，也應改變回答嗎？

沒有圖片時應直接走原文字路徑，不多加placeholder或改position。先檢查同權重的文字logits完全一樣，才比較多模態微調後能力。

**把直覺寫成數學：**同一weights、相同ids、相同positions下，兩個等價入口應相同。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.model import TinyLM, ModelConfig
from tiny_perceptron.multimodal import MultiModalLM

lm = TinyLM(ModelConfig(width=8))
multimodal = MultiModalLM(lm)
ids = torch.tensor([1, 20, 30])
a = lm(ids[None])["logits"]
b = multimodal(ids)["logits"]
assert torch.equal(a, b)
print("純文字路徑一致")

**如何讀結果：**這只驗證入口包裝，不能代替微調後的文字retention測試。

**只改一件事：**只故意加一個圖片標記但不提供圖，確認應拒絕此資料。
